# Privacy, PII and Data Governance Analysis
## NovaCred Credit Application Dataset - Governance Officer 

#### We will, in the following notebook, deeply analyse the potential risks for privacy and gorvernance of the NovaCred credit application dataset. By setting ourselves different objectives such as pin pointing personal identifiable informations, map findings to GDPR requirements and more, we will aim to propose governance improvements for the NovaCard system. 

In [13]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, HTML

# Define file path
data_path = Path("../Data/raw_credit_applications.json")

# Load raw JSON file
with open(data_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(raw_data)

# Basic preview
print(f"Number of records: {len(df)}")
print(f"Columns: {list(df.columns)}")

df.head()

Number of records: 502
Columns: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp', 'loan_purpose', 'notes']


,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


#### We flatten the data set, as it contains a nested structure, to use it in a more effective way accross the rest of our analyses. 

In [7]:
applicant_df = pd.json_normalize(df["applicant_info"]).add_prefix("applicant_")
financials_df = pd.json_normalize(df["financials"]).add_prefix("financial_")
decision_df = pd.json_normalize(df["decision"]).add_prefix("decision_")

# Combine into a single dataframe
df_flat = pd.concat(
    [
        df["_id"],
        applicant_df,
        financials_df,
        decision_df,
        df["spending_behavior"],
        df["processing_timestamp"],
        df["loan_purpose"],
        df["notes"],
    ],
    axis=1,
)

# Inspect result
df_flat.head()

,_id,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_gender,applicant_date_of_birth,applicant_zip_code,financial_annual_income,financial_credit_history_months,...,financial_savings_balance,financial_annual_salary,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,spending_behavior,processing_timestamp,loan_purpose,notes
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,...,31212,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,...,17915,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,NaN,NaN
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,...,37909,NaN,True,NaN,3.7,59000.0,"[{'category': 'Rent', 'amount': 109}]",NaN,vacation,NaN
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,...,0,NaN,True,NaN,4.3,34000.0,"[{'category': 'Fitness', 'amount': 575}]",NaN,NaN,NaN
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,...,31763,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,NaN,NaN


In [9]:
# Visualization setup for governance reporting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Simple governance-oriented color palette
GOV_COLORS = {
    "critical": "#B03A2E",   # dark red
    "high": "#D35400",       # orange
    "medium": "#CA8A04",     # amber
    "low": "#2E86AB",        # blue
    "ok": "#2E8B57",         # green
    "text": "#2C3E50",       # dark gray-blue
    "grid": "#E5E7EB",       # light gray
    "bg": "#FAFAFA"          # soft background
}

# Global plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.facecolor": GOV_COLORS["bg"],
    "axes.facecolor": "white",
    "axes.edgecolor": "#D0D7DE",
    "axes.labelcolor": GOV_COLORS["text"],
    "axes.titlecolor": GOV_COLORS["text"],
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "xtick.color": GOV_COLORS["text"],
    "ytick.color": GOV_COLORS["text"],
    "grid.color": GOV_COLORS["grid"],
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
    "font.size": 11,
    "legend.frameon": False
})

print("Governance visualization style loaded.")

Governance visualization style loaded.


## PII Identification 

In [12]:
pii_sample = df_flat[
    [
        "applicant_full_name",
        "applicant_email",
        "applicant_ssn",
        "applicant_ip_address",
        "applicant_date_of_birth",
        "applicant_gender",
        "applicant_zip_code"
    ]
].head()

pii_sample

,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_date_of_birth,applicant_gender,applicant_zip_code
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,2001-03-09,Male,10036
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,1992-03-31,M,10032
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,1989-10-24,Male,10075
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,1983-04-25,Male,10077
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,1999-05-21,M,10080


In [15]:
# --- PII inventory with GDPR mapping ---

def highlight_risk(val):
    colors = {
        "Critical": "background-color:#F8D7DA; color:#721C24; font-weight:bold;",
        "High": "background-color:#FDE2C8; color:#7C2D12;",
        "Medium": "background-color:#FFF3CD; color:#664D03;",
        "Low": "background-color:#D1E7DD; color:#0F5132;"
    }
    return colors.get(val, "")

pii_inventory = pd.DataFrame({
    "field_name": [
        "applicant_full_name",
        "applicant_email",
        "applicant_ssn",
        "applicant_ip_address",
        "applicant_date_of_birth",
        "applicant_gender",
        "applicant_zip_code"
    ],
    "pii_category": [
        "Direct identifier",
        "Direct identifier",
        "Sensitive identifier",
        "Online identifier",
        "Personal / demographic data",
        "Protected attribute",
        "Quasi-identifier"
    ],
    "risk_level": [
        "High",
        "High",
        "Critical",
        "High",
        "Medium",
        "High",
        "Medium"
    ],
    "gdpr_article_or_principle": [
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(c), 5(1)(f) - Minimization & confidentiality",
        "Art. 4(1), 5(1)(f) - Personal data & confidentiality",
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(a), Art. 22 - Fairness & automated decision-making",
        "Art. 5(1)(c), Art. 22 - Minimization & discrimination risk"
    ],
    "governance_risk": [
        "Directly identifies the applicant",
        "Directly identifies the applicant and exposes contact information",
        "Highly sensitive identifier that should not be stored in plain text",
        "Can be linked to an individual or device context",
        "Can support re-identification when combined with other fields",
        "May create fairness and discrimination risk in lending decisions",
        "May act as a proxy for socioeconomic status or ethnicity"
    ]
})

pii_inventory.style.applymap(
    highlight_risk,
    subset=["risk_level"]
).set_properties(**{
    "text-align": "left"
})

/var/folders/6x/48zsnbjs4hz2kb565wlpx3800000gn/T/ipykernel_17573/3561871187.py:60: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  pii_inventory.style.applymap(


,field_name,pii_category,risk_level,gdpr_article_or_principle,governance_risk
0,applicant_full_name,Direct identifier,High,Art. 5(1)(c) - Data minimization,Directly identifies the applicant
1,applicant_email,Direct identifier,High,Art. 5(1)(c) - Data minimization,Directly identifies the applicant and exposes contact information
2,applicant_ssn,Sensitive identifier,Critical,"Art. 5(1)(c), 5(1)(f) - Minimization & confidentiality",Highly sensitive identifier that should not be stored in plain text
3,applicant_ip_address,Online identifier,High,"Art. 4(1), 5(1)(f) - Personal data & confidentiality",Can be linked to an individual or device context
4,applicant_date_of_birth,Personal / demographic data,Medium,Art. 5(1)(c) - Data minimization,Can support re-identification when combined with other fields
5,applicant_gender,Protected attribute,High,"Art. 5(1)(a), Art. 22 - Fairness & automated decision-making",May create fairness and discrimination risk in lending decisions
6,applicant_zip_code,Quasi-identifier,Medium,"Art. 5(1)(c), Art. 22 - Minimization & discrimination risk",May act as a proxy for socioeconomic status or ethnicity


#### As some of those informations are set between medium and critical risk level, we want to analyse to what extent those PII fileds are exposed and present in our data set. 

In [16]:
# Basic completeness and exposure analysis for key PII fields

pii_cols = [
    "applicant_full_name",
    "applicant_email",
    "applicant_ssn",
    "applicant_ip_address",
    "applicant_date_of_birth",
    "applicant_gender",
    "applicant_zip_code"
]

pii_summary = pd.DataFrame({
    "field_name": pii_cols,
    "non_null_count": [df_flat[col].notna().sum() for col in pii_cols],
    "missing_count": [df_flat[col].isna().sum() for col in pii_cols],
    "missing_pct": [round(df_flat[col].isna().mean() * 100, 2) for col in pii_cols],
    "unique_values": [df_flat[col].nunique(dropna=True) for col in pii_cols]
})

pii_summary

,field_name,non_null_count,missing_count,missing_pct,unique_values
0,applicant_full_name,502,0,0.0,475
1,applicant_email,502,0,0.0,494
2,applicant_ssn,497,5,1.0,494
3,applicant_ip_address,497,5,1.0,496
4,applicant_date_of_birth,501,1,0.2,494
5,applicant_gender,501,1,0.2,5
6,applicant_zip_code,501,1,0.2,196


#### We see that the PII filed create privacy, compliance and fairness risks for NovaCred. We have both direct and indirect identifiers in our categories. 
- **Direct identifiers** such as full name, email, and SSN can immediately identify an applicant.
- **Online identifiers** such as IP address are personal data under GDPR and may reveal device or location context.
- **Quasi-identifiers** such as date of birth and ZIP code may not identify someone alone, but can enable re-identification when combined with other fields.
- **Protected attributes** such as gender create additional fairness and discrimination risks in automated credit decisions.

#### We need to execute a brief deep dive into the SSN, as the sensitivity of it can lead to serious consequences such as financial fraud, or impersonation from outsiders if the data is leaked. We could assume that this violates GDPR data minimization as a machine learning model should not rquire SSN to predict credit worthiness (if not proved otherwise) 